In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
from PyPDF2 import PdfReader
from datetime import datetime
from pdfminer.pdfparser import PDFSyntaxError
import pdfplumber
from collections import defaultdict
import camelot as cam
import logging
import shutil

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
start = time.time()

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_colwidth', 1000)

BASE_DIR = "rbko_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

for subfolder in ['balance-sheet', 'income-statement', 'financial-indicators']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)

PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")
FILE_METADATA_LOG = os.path.join(BASE_DIR, "file_metadata.csv")

url_raiffeisen = 'https://www.raiffeisen-kosovo.com/sq/rreth-nesh/raporte-dhe-njoftime/pasqyrat-financiare.html'



def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()


def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)


def download_pdf(url, local_dir):
    """Download PDF to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    response = requests.get(url)
    with open(local_path, 'wb') as f:
        f.write(response.content)
    logging.info(f"Downloaded: {url} to {local_path}")
    return local_path


def scrape_pdf_links(page_url):
    """Scrape PDF links from the Raiffeisen page"""
    response = requests.get(page_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    pdf_links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.lower().endswith('.pdf'):
            full_url = urljoin(page_url, href)
            pdf_links.append(full_url)
    return pdf_links


def extract_date(text, filename):
    """Extract a date string from text or filename"""
    # Try to find date pattern like 31.12.2023 or 31/12/2023
    match = re.search(r'\d{1,2}[./]\d{1,2}[./]\d{4}', text)
    if match:
        date_str = match.group()
        # Convert to standard format with dots
        return date_str.replace('/', '.')
    
    match = re.search(r'\d{1,2}[./]\d{4}', text)
    if match:
        # Found MM.YYYY format, need to add day
        date_parts = re.split(r'[./]', match.group())
        month = date_parts[0].zfill(2)
        year = date_parts[1]
        
        # Determine day based on month
        if month in ['12', '03']:
            day = '31'
        else:
            day = '30'
        
        return f"{day}.{month}.{year}"
    
    # Try filename
    match = re.search(r'\d{1,2}[./-]\d{1,2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    return None


def download_all(force=False):
    """Download PDFs and categorize them locally.

    When ``force`` is True, ignore the processed-files log and download every
    link found on the page. Otherwise behave as before and skip names that
    already appear in the log.
    """
    processed = set() if force else load_processed_files()
    if force:
        logging.info("Force mode enabled: all files will be downloaded")
    pdf_links = scrape_pdf_links(url_raiffeisen)
    new_files = []
    file_metadata = []


    # Keywords for categorization
    Balance_sheet = [
        'balance%20', 'bsh_', 'bilanci', 'bilanc', 'bilance_', 'bilanc_sheet',
        'balance-sheet', 'balance_sheet', 'balancesheet', 'bgj'
    ]

    Income_Statement = [
        'pasqyra', 'is_', 'income_statement'
    ]

    Financial_Indicators = [
        'treguesitfinanciar', 'treguesit_financiar', 'indikatoret-financiar'
    ]

    categories = {
        'balance-sheet': Balance_sheet,
        'income-statement': Income_Statement,
        'financial-indicators': Financial_Indicators
    }

    for pdf_url in pdf_links:
        file_name = os.path.basename(pdf_url)
        
        # Skip if already processed
        if file_name in processed:
            logging.info(f"File '{file_name}' already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_pdf(pdf_url, ORIGINAL_DIR)



        # Try to read metadata creation date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                pdf_reader = PdfReader(f)
                metadata = pdf_reader.metadata
                for k, v in metadata.items():
                    if "/CreationDate" in k:
                        date_str = re.search(r'D:(\d+)', v).group(1)
                        dt = datetime.strptime(date_str, '%Y%m%d%H%M%S')
                        break
        except Exception as e:
            logging.info(f"Error reading date from metadata: {e}")

        # Categorize and rename file
        try:
            with pdfplumber.open(file_path) as pdf:
                full_text = ""
                for page in pdf.pages:
                    text = page.extract_text() or ""
                    full_text += text + "\n"
                
                date = extract_date(full_text, file_name)
                
                if date:
                    # Determine category based on filename
                    file_lower = file_name.lower()
                    found_category = None
                    
                    for category, keywords in categories.items():
                        if any(keyword in file_lower for keyword in keywords):
                            found_category = category
                            break
                    
                    if found_category:
                        if found_category == 'balance-sheet':
                            new_name = f'rbko_bs_{date}.pdf'
                        elif found_category == 'income-statement':
                            new_name = f'rbko_is_{date}.pdf'
                        else:
                            new_name = f'rbko_fi_{date}.pdf'
                        
                        # Copy to categorized folder
                        dest_dir = os.path.join(CURRENT_DIR, found_category)
                        dest_path = os.path.join(dest_dir, new_name)
                        shutil.copy2(file_path, dest_path)
                        
                        file_metadata.append({
                            'file_name': file_name,
                            'file_path': file_path,
                            'current_name': new_name,
                            'category': found_category,
                            'reporting_date': date,
                            'creation_date': dt,
                            'download_date': datetime.now(),
                            'is_corrupt': 0
                        })
                        
                        logging.info(f"Categorized {file_name} as {found_category} -> {new_name}")

                    else:
                        # Could not categorize
                        file_metadata.append({
                            'file_name': file_name,
                            'file_path': file_path,
                            'current_name': None,
                            'category': 'uncategorized',
                            'reporting_date': date,
                            'creation_date': dt,
                            'download_date': datetime.now(),
                            'is_corrupt': 0
                        })
                else:
                    # Could not extract date
                    file_metadata.append({
                        'file_name': file_name,
                        'file_path': file_path,
                        'current_name': None,
                        'category': 'no-date',
                        'reporting_date': None,
                        'creation_date': dt,
                        'download_date': datetime.now(),
                        'is_corrupt': 0
                    })
                    
        except Exception as e:
            logging.error(f"Error processing {file_name}: {e}")
            file_metadata.append({
                'file_name': file_name,
                'file_path': file_path,
                'current_name': None,
                'category': 'error',
                'reporting_date': None,
                'creation_date': dt,
                'download_date': datetime.now(),
                'is_corrupt': 1
            })
        
        # Mark as processed
        save_processed_file(file_name)

    # Save metadata to CSV
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        
        # Append to main metadata log
        if os.path.exists(FILE_METADATA_LOG):
            existing = pd.read_csv(FILE_METADATA_LOG)
            metadata_df = pd.concat([existing, metadata_df], ignore_index=True)
        metadata_df.to_csv(FILE_METADATA_LOG, index=False)
        
        logging.info(f"File metadata saved to {metadata_path}")

    if new_files:
        logging.info(f"Downloaded {len(new_files)} new files.")
    else:
        logging.info("No new files detected.")

    return [m.get('current_name') for m in file_metadata if m.get('current_name')]


def replace_negatives(value):
    """Convert parenthesized numbers to negative format"""
    if pd.isna(value) or value == '-':
        return '0'
    value = str(value)
    if '(' in value and ')' in value:
        return '-' + value.replace('(', '').replace(')', '').replace(',', '')
    else:
        return value


def read_income(pdf_path, filename):
    """Read income statement from a local PDF file"""
    try:
        tables = cam.read_pdf(pdf_path, flavor='stream')
        income_dfs = []
        for table in tables:
            df = table.df
            income_dfs.append(df)

        income_statement = pd.concat(income_dfs, ignore_index=True)
        
        # Handle different possible structures
        if income_statement.shape[1] >= 3:
            income_statement = income_statement.iloc[:, :3]
            income_statement.columns = ['Category', 'Previous Quarter', 'Current Quarter']
            
            # Clean up rows if needed
            if income_statement.shape[0] > 2 and str(income_statement.iloc[1, 0]) == '':
                income_statement = income_statement.iloc[3:].reset_index(drop=True)
            
            income_statement['Category'] = income_statement['Category'].str.replace('\n', ' ')
            income_statement['Category'] = income_statement['Category'].str.strip()
            income_statement['FILE_NAME'] = filename
            
            return income_statement
        else:
            logging.warning(f"Income statement PDF {filename} has unexpected structure")
            return pd.DataFrame()
            
    except Exception as e:
        logging.error(f"Error reading income statement {filename}: {e}")
        return pd.DataFrame()


def read_balance(pdf_path, filename):
    """Read balance sheet from a local PDF file"""
    try:
        tables = cam.read_pdf(pdf_path, flavor='stream')
        balance_dfs = []
        for table in tables:
            df = table.df
            balance_dfs.append(df)

        balance_sheet = pd.concat(balance_dfs, ignore_index=True)
        
        # Handle different possible structures
        if balance_sheet.shape[1] >= 3:
            balance_sheet = balance_sheet.iloc[:, :3]
            balance_sheet.columns = ['Category', 'Previous Quarter', 'Current Quarter']
            
            # Clean up rows if needed (drop header rows and empty rows)
            balance_sheet = balance_sheet[balance_sheet['Category'].str.strip() != '']
            balance_sheet = balance_sheet.reset_index(drop=True)
            
            balance_sheet['Category'] = balance_sheet['Category'].str.replace('\n', ' ')
            balance_sheet['Category'] = balance_sheet['Category'].str.strip()
            balance_sheet['FILE_NAME'] = filename
            
            return balance_sheet
        else:
            logging.warning(f"Balance sheet PDF {filename} has unexpected structure")
            return pd.DataFrame()
            
    except Exception as e:
        logging.error(f"Error reading balance sheet {filename}: {e}")
        return pd.DataFrame()


def extract_text_from_pdf(pdf_path):
    """Extract text from PDF for date extraction"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages[:3]:  # Check first 3 pages for date
                page_text = page.extract_text() or ""
                text += page_text + "\n"
            return text
    except Exception as e:
        logging.error(f"Error extracting text from {pdf_path}: {e}")
        return ""


def gather_current_files():
    """Return list of PDF names currently stored in the categorized folders."""
    files = []
    for category in ['income-statement', 'balance-sheet']:
        category_dir = os.path.join(CURRENT_DIR, category)
        if os.path.exists(category_dir):
            files.extend([fn for fn in os.listdir(category_dir) if fn.lower().endswith('.pdf')])
    return files


def main(force=False, extract_only=False):
    """Main execution function.

    ``force`` is passed through to :func:`download_all`.
    If ``extract_only`` is True the function skips scraping/downloading and
    simply processes whatever PDF filenames currently exist under
    ``CURRENT_DIR`` (useful when you already have the files and just want to
    convert them to CSV).
    """
    # decide which files to treat as "new"
    if extract_only:
        logging.info("Extract-only mode: processing existing files without downloading")
        new_files = gather_current_files()
    else:
        new_files = download_all(force=force)
    logging.info(f"NEW FILES: {new_files}")
    
    if new_files:
        # Collect all current files that are new
        is_dfs = []
        bs_dfs = []
        
        # Walk through current directory subfolders
        for category in ['income-statement', 'balance-sheet']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                for filename in os.listdir(category_dir):
                    if filename in new_files:
                        file_path = os.path.join(category_dir, filename)
                        
                        if category == 'income-statement':
                            try:
                                df = read_income(file_path, filename)
                                if not df.empty:
                                    # Extract date from filename or content
                                    date_match = re.search(r'(\d{2}\.\d{2}\.\d{4})', filename)
                                    if date_match:
                                        date = date_match.group(1)
                                    else:
                                        text = extract_text_from_pdf(file_path)
                                        date = extract_date(text, filename)
                                    
                                    if date:
                                        df['DT_REPORT'] = date
                                        is_dfs.append(df)
                                        logging.info(f"Processed income statement: {filename}")
                            except Exception as e:
                                logging.error(f"Error processing income statement {filename}: {e}")
                                
                        elif category == 'balance-sheet':
                            try:
                                df = read_balance(file_path, filename)
                                if not df.empty:
                                    # Extract date from filename or content
                                    date_match = re.search(r'(\d{2}\.\d{2}\.\d{4})', filename)
                                    if date_match:
                                        date = date_match.group(1)
                                    else:
                                        text = extract_text_from_pdf(file_path)
                                        date = extract_date(text, filename)
                                    
                                    if date:
                                        df['DT_REPORT'] = date
                                        bs_dfs.append(df)
                                        logging.info(f"Processed balance sheet: {filename}")
                            except Exception as e:
                                logging.error(f"Error processing balance sheet {filename}: {e}")
        
        # Combine all dataframes
        full_is = pd.concat(is_dfs, ignore_index=True) if is_dfs else pd.DataFrame()
        full_bs = pd.concat(bs_dfs, ignore_index=True) if bs_dfs else pd.DataFrame()
        
        if full_is.empty and full_bs.empty:
            logging.info("No data extracted from files")
            return "Request processed successfully with no new data."
        
        # Process Income Statement
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'FILE_NAME', 'DT_REPORT']
            full_is['BANK_ID'] = 8
            full_is['DT_REPORT'] = full_is['DT_REPORT'].str.replace('.', '/')
            
            # Clean values
            full_is['PREVIOUS_INCOME_VALUE'] = full_is['PREVIOUS_INCOME_VALUE'].astype(str).str.replace('-', '0')
            full_is['CURRENT_INCOME_VALUE'] = full_is['CURRENT_INCOME_VALUE'].astype(str).str.replace('-', '0')
            
            full_is['PREVIOUS_INCOME_VALUE'] = full_is['PREVIOUS_INCOME_VALUE'].apply(replace_negatives)
            full_is['CURRENT_INCOME_VALUE'] = full_is['CURRENT_INCOME_VALUE'].apply(replace_negatives)
            
            # Remove commas and convert to numeric
            full_is['PREVIOUS_INCOME_VALUE'] = pd.to_numeric(
                full_is['PREVIOUS_INCOME_VALUE'].astype(str).str.replace(',', ''), 
                errors='coerce'
            ).fillna(0)
            full_is['CURRENT_INCOME_VALUE'] = pd.to_numeric(
                full_is['CURRENT_INCOME_VALUE'].astype(str).str.replace(',', ''), 
                errors='coerce'
            ).fillna(0)
            
            # Category mapping for Income Statement
            replacement_map = {
                'Të hyrat nga interesi': '1',
                'Shpenzimet e interesit': '2',
                'Neto të hyrat nga interesi': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Gjithsej të hyrat': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Fitimi (humbja) para tatimit': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Fitimi (humbja) neto': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19',
                'Interest income': '1',
                'Interest expenses': '2',
                'Net interest income': '3',
                'Fee and commission income': '4',
                'Fee and commission expenses': '5',
                'Net fee and commission income': '6',
                'Foreign exchange gains/(losses)': '7',
                'Net gains from other financial instruments': '8',
                'Neto other operating income (expenses)': '9',
                'Total Income': '10',
                'Loan loss provision charges': '11',
                'Net profit before taxes': '14',
                'Income tax expenses': '15',
                'Net profit for the period': '16',
                'Other comprehensive income': '17',
                'Total other comprehensive income': '19'
            }
            
            full_is['INCOME_CATEGORY'] = full_is['INCOME_CATEGORY'].replace(replacement_map)
            full_is['INCOME_CATEGORY'] = pd.to_numeric(full_is['INCOME_CATEGORY'], errors='coerce')
            full_is = full_is.dropna(subset=['INCOME_CATEGORY'])
            
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], format='%d/%m/%Y', errors='coerce')
            full_is['CURR_ID'] = 1
            full_is = full_is.fillna(0)
        
        # Process Balance Sheet
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'FILE_NAME', 'DT_REPORT']
            full_bs['BANK_ID'] = 8
            full_bs['DT_REPORT'] = full_bs['DT_REPORT'].str.replace('.', '/')
            
            full_bs['BALANCE_CATEGORY'] = full_bs['BALANCE_CATEGORY'].str.replace('\n', ' ')
            
            # Clean values
            full_bs['PREVIOUS_BALANCE_VALUE'] = full_bs['PREVIOUS_BALANCE_VALUE'].astype(str).str.replace('-', '0')
            full_bs['CURRENT_BALANCE_VALUE'] = full_bs['CURRENT_BALANCE_VALUE'].astype(str).str.replace('-', '0')
            
            full_bs['PREVIOUS_BALANCE_VALUE'] = full_bs['PREVIOUS_BALANCE_VALUE'].apply(replace_negatives)
            full_bs['CURRENT_BALANCE_VALUE'] = full_bs['CURRENT_BALANCE_VALUE'].apply(replace_negatives)
            
            # Remove commas and convert to numeric
            full_bs['PREVIOUS_BALANCE_VALUE'] = pd.to_numeric(
                full_bs['PREVIOUS_BALANCE_VALUE'].astype(str).str.replace(',', ''), 
                errors='coerce'
            ).fillna(0)
            full_bs['CURRENT_BALANCE_VALUE'] = pd.to_numeric(
                full_bs['CURRENT_BALANCE_VALUE'].astype(str).str.replace(',', ''), 
                errors='coerce'
            ).fillna(0)
            
            # Category mapping for Balance Sheet
            replacement_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Kërkesat ndaj bankave': '21',
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë tatimore të shtyra': '29',
                'Pasuritë tjera': '30',
                'Gjithsej pasuritë': '31',
                'Depozitat e klientëve': '32',
                'Detyrimet ndaj bankave': '33',
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35',
                'Detyrimet tjera': '36',
                'Gjithsej detyrimet': '37',
                'Kapitali aksionar': '38',
                'Rezervat e kapitalit': '40',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi/(humbja) e vitit aktual': '42',
                'Përbërësit tjerë të ekuititetit': '43',
                'Gjithsej ekuititeti i aksionarëve': '44',
                'Gjithsej detyrimet dhe ekuititeti i aksionarëve': '45'
            }
            
            full_bs['BALANCE_CATEGORY'] = full_bs['BALANCE_CATEGORY'].replace(replacement_map)
            full_bs['BALANCE_CATEGORY'] = pd.to_numeric(full_bs['BALANCE_CATEGORY'], errors='coerce')
            full_bs = full_bs.dropna(subset=['BALANCE_CATEGORY'])
            
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], format='%d/%m/%Y', errors='coerce')
            full_bs['CURR_ID'] = 1
            full_bs = full_bs.fillna(0)
        
        # Get latest reports only
        if not full_is.empty and 'DT_REPORT' in full_is.columns:
            try:
                full_is.sort_values(by='DT_REPORT', ascending=False, inplace=True)
                max_date_is = full_is['DT_REPORT'].max()
                full_is = full_is[full_is['DT_REPORT'] == max_date_is]
                full_is['DT_REPORT'] = full_is['DT_REPORT'].dt.strftime('%Y%m%d').astype(int)
            except Exception as e:
                logging.info(f"Failed to sort income statements by DT_REPORT: {e}")
        
        if not full_bs.empty and 'DT_REPORT' in full_bs.columns:
            try:
                full_bs.sort_values(by='DT_REPORT', ascending=False, inplace=True)
                max_date_bs = full_bs['DT_REPORT'].max()
                full_bs = full_bs[full_bs['DT_REPORT'] == max_date_bs]
                full_bs['DT_REPORT'] = full_bs['DT_REPORT'].dt.strftime('%Y%m%d').astype(int)
            except Exception as e:
                logging.info(f"Failed to sort balance sheets by DT_REPORT: {e}")
        
        # Save outputs locally
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Save to CSV
        if not full_is.empty:
            is_output_path = os.path.join(OUTPUT_DIR, f"rbko_income_statement_{timestamp}.csv")
            full_is.to_csv(is_output_path, index=False)
            logging.info(f"Income Statement saved to {is_output_path}")
        
        if not full_bs.empty:
            bs_output_path = os.path.join(OUTPUT_DIR, f"rbko_balance_sheet_{timestamp}.csv")
            full_bs.to_csv(bs_output_path, index=False)
            logging.info(f"Balance Sheet saved to {bs_output_path}")
        
        # Save to Excel
        excel_path = os.path.join(OUTPUT_DIR, f"rbko_financial_report_{timestamp}.xlsx")
        with pd.ExcelWriter(excel_path) as writer:
            if not full_is.empty:
                full_is.to_excel(writer, sheet_name='Income Statement', index=False)
            if not full_bs.empty:
                full_bs.to_excel(writer, sheet_name='Balance Sheet', index=False)
        
        logging.info(f"Excel report saved to {excel_path}")
        logging.info(f"Processing completed in {time.time() - start:.2f} seconds")
        
        return "Request processed successfully with new data."
    
    return "Request processed successfully with no new data."


if __name__ == "__main__":
    import sys
    force_flag = '--force' in sys.argv or '-f' in sys.argv
    extract_flag = '--extract' in sys.argv or '-e' in sys.argv
    result = main(force=force_flag, extract_only=extract_flag)
    print(result)